# HunyuanVideo 1.5 — Colab T4

**Быстрая генерация видео из текста** с HunyuanVideo 1.5 (Tencent) на бесплатном T4 GPU.

```
[1. GPU] → [2. Установка] → [3. Модели] → [4. Воркфлоу] → [5. Медиа] → [6. Запуск]
   |            |                |              |              |             |
 Проверка    ComfyUI +      ~36 ГБ BF16     Скачивание    Опционально    Cloudflare
  CUDA      кастомные       трансформер    воркфлоу T2V   (для I2V)      туннель
            ноды         + энкодеры + VAE
```

---

### Когда выбирать этот ноутбук?

Используйте **HunyuanVideo**, если вам нужно:
- Быстро проверить идею промпта (в ~2 раза быстрее Wan)
- Генерировать видео **только из текста** (T2V) без референсных изображений
- Экономить VRAM — пиковое потребление ~12 ГБ (vs ~15 ГБ у Wan)
- Итерировать по промптам перед финальным рендером в Wan

Используйте **Wan 2.2** (`colab_wan_video.ipynb`), если вам нужно:
- Максимальное качество видео
- Режимы I2V, V2V, говорящая голова, длинные видео (20-30 сек)
- Больше контроля через VACE и дополнительные воркфлоу

### Сравнение моделей

| | HunyuanVideo 1.5 | Wan 2.2 14B |
|---|---|---|
| **Размер модели** | ~25.6 ГБ (BF16, загрузка FP8) | ~14 ГБ (FP8) |
| **Скорость генерации** | ~2x быстрее | Базовая |
| **Качество** | Отличное, точное следование тексту | Лучшее в целом |
| **Пик VRAM** | ~12 ГБ | ~15 ГБ |
| **Режимы** | T2V | T2V, I2V, V2V, Talking, Reels |
| **FPS выхода** | 16 fps | 24 fps |
| **Лучше для** | Быстрых итераций, экспериментов | Финального качества, сложных задач |

> **Совет:** Начните с HunyuanVideo для поиска идеального промпта, затем переключитесь на Wan для финального рендера.

> **Примечание:** Модели загружаются в формате BF16, но ComfyUI автоматически конвертирует трансформер в FP8 при загрузке, что позволяет уместиться в 16 ГБ VRAM на T4.

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!test -d ComfyUI-VideoHelperSuite || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git

!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-KJNodes/requirements.txt -q -c /tmp/numpy_constraint.txt

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей HunyuanVideo (~36 ГБ)
import os, shutil

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

# Проверка свободного места на диске
free_gb = shutil.disk_usage("/content").free / 1024**3
print(f"Свободно на диске: {free_gb:.1f} ГБ (нужно ~36 ГБ)")
if free_gb < 38:
    print("  ВНИМАНИЕ: Мало места! Модели могут не поместиться.")
    print("  Совет: Перезапустите рантайм (Runtime -> Disconnect and delete runtime)")

files = {
    # HunyuanVideo T2V transformer BF16 (~25.6 ГБ) — загружается как FP8 через воркфлоу
    f"{M}/diffusion_models/hunyuan_video_t2v_720p_bf16.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/diffusion_models/hunyuan_video_t2v_720p_bf16.safetensors",
    # LLM текстовый энкодер FP8 (~9.09 ГБ)
    f"{M}/text_encoders/llava_llama3_fp8_scaled.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/llava_llama3_fp8_scaled.safetensors",
    # CLIP текстовый энкодер (~246 МБ)
    f"{M}/text_encoders/clip_l.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/text_encoders/clip_l.safetensors",
    # VAE (~493 МБ)
    f"{M}/vae/hunyuan_video_vae_bf16.safetensors":
        "https://huggingface.co/Comfy-Org/HunyuanVideo_repackaged/resolve/main/split_files/vae/hunyuan_video_vae_bf16.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
all_ok = True
for dest in files:
    if not os.path.exists(dest) or os.path.getsize(dest) < 1024:
        if os.path.exists(dest):
            os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")
        all_ok = False

if all_ok:
    print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_hunyuan_t2v.json" "{RAW}/video_hunyuan_t2v.json"
print("Воркфлоу скачан!")

In [ ]:
#@title 5. Загрузка медиа (опционально)
#@markdown ### Эта ячейка опциональна
#@markdown HunyuanVideo — это прежде всего **Text-to-Video** модель.
#@markdown Загружать медиа нужно только если вы планируете использовать
#@markdown экспериментальные I2V воркфлоу в будущем.
#@markdown
#@markdown **Поддерживаемые форматы:** `.png`, `.jpg`, `.jpeg`, `.webp`
#@markdown
#@markdown Если вам нужен только T2V — **пропустите эту ячейку** и переходите к шагу 6.

from google.colab import files
from IPython.display import display, Image as IPImage
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

ALLOWED_EXT = {".png", ".jpg", ".jpeg", ".webp"}

uploaded = files.upload()

if not uploaded:
    print("Файлы не загружены. Это нормально — для T2V медиа не нужно.")
else:
    saved = []
    skipped = []

    for name, data in uploaded.items():
        ext = os.path.splitext(name)[1].lower()
        if ext not in ALLOWED_EXT:
            skipped.append(f"  {name} (формат {ext} не поддерживается)")
            continue
        path = os.path.join(INPUT_DIR, name)
        with open(path, "wb") as f:
            f.write(data)
        saved.append(name)

        # Превью загруженного изображения
        try:
            display(IPImage(filename=path, width=300))
        except Exception:
            pass

    # Итог загрузки
    print(f"\n{'='*50}")
    print(f"РЕЗУЛЬТАТ ЗАГРУЗКИ")
    print(f"{'='*50}")

    if saved:
        print(f"Сохранено файлов: {len(saved)}")
        for name in saved:
            size_kb = len(uploaded[name]) / 1024
            print(f"  {name} ({size_kb:.0f} КБ)")

    if skipped:
        print(f"\nПропущено (неподдерживаемый формат): {len(skipped)}")
        for msg in skipped:
            print(msg)
        print(f"  Допустимые форматы: {', '.join(sorted(ALLOWED_EXT))}")

    print(f"\nВсе файлы в input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Запуск ComfyUI
#@markdown ---
#@markdown ### Способ подключения
#@markdown ngrok — самый стабильный. Cloudflare — без регистрации. Localtunnel — запасной.
tunnel_method = "ngrok" #@param ["ngrok", "cloudflare", "localtunnel"]
ngrok_token = "" #@param {type:"string"}
#@markdown > Токен ngrok: [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

import subprocess, time, re, os, requests, shutil

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
LOG_FILE = "/content/comfyui.log"

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--enable-cors-header", "*"],
    cwd="/content/ComfyUI",
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
started = False
for i in range(24):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8188/api/system_stats", timeout=3).status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            started = True
            break
    except Exception:
        pass

if not started:
    print("  ComfyUI не ответил за 120 сек.")
    with open(LOG_FILE) as f:
        lines = f.readlines()
        errors = [l for l in lines if any(w in l.lower() for w in ['error', 'traceback', 'exception', 'failed'])]
        if errors:
            print("  --- Ошибки ---")
            for l in errors[-15:]:
                print(f"  {l.rstrip()}")
        for l in lines[-10:]:
            print(f"  {l.rstrip()}")
    raise RuntimeError("ComfyUI не запустился. Проверьте ошибки выше.")

with open(LOG_FILE) as f:
    log_text = f.read()
import_errors = [l for l in log_text.split('\n') if 'cannot import' in l.lower() or 'no module named' in l.lower()]
if import_errors:
    print(f"\n  Предупреждения при загрузке нод ({len(import_errors)}):")
    for l in import_errors[:5]:
        print(f"    {l.strip()}")

url = None
if tunnel_method == "ngrok":
    !pip install pyngrok -q
    from pyngrok import ngrok
    if not ngrok_token:
        import getpass
        ngrok_token = getpass.getpass("Введите ngrok authtoken (dashboard.ngrok.com → Your Authtoken): ")
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8188)
    url = tunnel.public_url

elif tunnel_method == "cloudflare":
    if not shutil.which("cloudflared"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb 2>&1 | tail -1
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break

elif tunnel_method == "localtunnel":
    !npm install -g localtunnel > /dev/null 2>&1
    tunnel = subprocess.Popen(
        ["lt", "--port", "8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(8)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.loca\.lt", line)
        if match:
            url = match.group(0)
            break

if url:
    print(f"\n{'='*60}")
    print(f"  ComfyUI готов! Откройте: {url}")
    print(f"{'='*60}")
    if tunnel_method == "ngrok":
        print(f"\n  ngrok — стабильное соединение, без белых страниц.")
    if tunnel_method == "localtunnel":
        print(f"\n  Localtunnel попросит пароль — нажмите ссылку на странице.")
    print(f"\n  Загрузите воркфлоу: Меню -> Load -> video_hunyuan_t2v")
    print(f"\n  Если страница белая:")
    print(f"    1. Обновите страницу (Ctrl+R)")
    print(f"    2. Если не помогло — попробуйте другой tunnel_method")
    print(f"\n  Логи ComfyUI: !cat {LOG_FILE}")
else:
    print(f"\nURL туннеля не найден. Попробуйте другой tunnel_method.")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/hunyuan"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Скопировано: {os.path.basename(v)}")
    print(f"\n{len(videos)} файлов сохранено в {dst}")

---
## Руководство по использованию

### Пошаговая инструкция

**Шаг 1.** Откройте ссылку на туннель из ячейки 6
  - Если ссылка не появилась, подождите 10-15 секунд и перезапустите ячейку

**Шаг 2.** Загрузите воркфлоу в ComfyUI
  - Нажмите **Меню** (три полоски слева вверху) -> **Load** -> выберите **video_hunyuan_t2v**

**Шаг 3.** Напишите промпт
  - Найдите текстовую ноду и замените промпт на свой
  - HunyuanVideo **очень точно** следует тексту — чем детальнее, тем лучше

**Шаг 4.** Настройте параметры (опционально)
  - Разрешение и количество кадров можно менять в нодах воркфлоу

**Шаг 5.** Нажмите **Queue Prompt** и ждите результат

---

### Пресеты разрешений (16:9)

| Размер | Кадры | Длительность | VRAM | Рекомендация |
|--------|-------|-------------|------|--------------|
| 848x480 | 61 | ~4 сек | ~12 ГБ | Быстрый тест |
| 848x480 | 97 | ~6 сек | ~14 ГБ | Оптимальный баланс |
| 640x368 | 129 | ~8 сек | ~13 ГБ | Максимальная длина |

> **Важно:** На T4 держите количество кадров ниже 129, чтобы избежать ошибки нехватки памяти (OOM).

---

### Советы для лучших результатов

1. **Пишите детальные промпты на английском.** HunyuanVideo лучше всего работает с описательными промптами на естественном языке

2. **Описывайте действие, камеру и свет.** Модель хорошо понимает кинематографические инструкции

3. **Начинайте с коротких видео (61 кадр).** Проверьте промпт, затем увеличивайте длительность

4. **Выход — 16 fps** (у Wan — 24 fps). Учитывайте это при планировании монтажа

5. **Если видео мутное или некачественное** — попробуйте увеличить steps в сэмплере до 25-30

### Пример хорошего промпта

> "A golden retriever running through a sunlit meadow, wildflowers swaying in the breeze, cinematic camera tracking shot, warm afternoon light, shallow depth of field, 4K quality"

### Пример плохого промпта

> "dog in field"

Чем больше деталей (субъект, действие, освещение, камера, настроение) — тем точнее результат.